# Kaggle Outline → BatikCraft Asset

Mengubah dataset wayang/ukiran/ornamen menjadi PNG outline transparan, `.batikasset` individual, dan `.batikpack` yang bisa diinstal di BatikCraftStudio.

Schema mengikuti aplikasi:
- `.batikasset`: JSON UTF-8 + `png_base64`
- `.batikpack`: ZIP berisi `manifest.json`, `assets/`, dan `thumbnails/`

In [ ]:
from __future__ import annotations
import base64, hashlib, io, json, re, zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import cv2
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display

# ===== CONFIG =====
INPUT_DIR = Path("/kaggle/input")  # ganti ke folder dataset spesifik
OUT = Path("/kaggle/working/batikcraft_outline_assets")
MAX_IMAGES = None                  # contoh: 20 untuk uji
MASTER_SIZE, THUMB_SIZE = 1024, 192
MODE = "hybrid"                    # hybrid | canny | adaptive
CANNY_LOW, CANNY_HIGH = 45, 145
ADAPTIVE_BLOCK, ADAPTIVE_C = 31, 11
BG_TOLERANCE, MIN_AREA = 30, 18
LINE_RGBA = (25, 18, 14, 255)
PACK_ID = "batikcraft-outline-collection"
PACK_NAME = "BatikCraft Outline Collection"
PACK_VERSION = "1.0.0"
PACK_AUTHOR = "Balya Rochmadi"
PACK_DESCRIPTION = "Outline wayang, ukiran, ornamen, motif, dan isen-isen dari Kaggle."

ASSET_FORMAT, ASSET_SCHEMA = "batikcraft-asset", "1.0"
PACK_FORMAT, PACK_SCHEMA = "batikcraft-asset-pack", "1.0"
CATEGORIES = ("motif-pokok", "isen-isen", "ornamen", "tekstur", "lainnya")
EXTS = {".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"}

for name in ("png","thumbs","previews","batikasset","batikpack"):
    (OUT/name).mkdir(parents=True, exist_ok=True)
print("Output:", OUT)

In [ ]:
# ruff: noqa: E701, E702, UP017
def safe_id(value: object) -> str:
    text = re.sub(r"[^a-z0-9._-]+","-",str(value).strip().casefold().replace(" ","-")).strip("-.")
    if not text: raise ValueError("ID kosong")
    return text[:120]

def png_bytes(img: Image.Image) -> bytes:
    b=io.BytesIO(); img.save(b,"PNG",optimize=True); return b.getvalue()

def discover(root: Path) -> list[Path]:
    files=sorted((p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXTS),
                 key=lambda p:p.as_posix().casefold())
    return files if MAX_IMAGES is None else files[:int(MAX_IMAGES)]

def infer(path: Path):
    t=" ".join(x.casefold() for x in path.parts)
    if any(x in t for x in ("isen","cecek","sawut")):
        return "isen-isen",("outline","line-art","isen-isen")
    if any(x in t for x in ("motif","batik","parang","kawung")):
        return "motif-pokok",("outline","line-art","motif-pokok")
    if any(x in t for x in ("texture","tekstur","grain")):
        return "tekstur",("outline","tekstur")
    return "ornamen",("outline","line-art","ornamen")

def name_of(path: Path) -> str:
    return re.sub(r"\s+"," ",re.sub(r"[_-]+"," ",path.stem)).strip().title()[:120] or "Asset Outline"

def clean_components(mask: np.ndarray) -> np.ndarray:
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8)*255,8)
    out=np.zeros_like(mask,dtype=np.uint8)
    for i in range(1,n):
        if int(stats[i,cv2.CC_STAT_AREA])>=MIN_AREA: out[lab==i]=255
    return out

def foreground(img: Image.Image) -> np.ndarray:
    a=np.asarray(img.convert("RGBA")); alpha=a[:,:,3]
    if int(alpha.min())<250 and int(alpha.max())>0:
        mask=(alpha>8).astype(np.uint8)*255
    else:
        rgb=a[:,:,:3].astype(np.float32); h,w=rgb.shape[:2]; q=max(1,min(16,h,w))
        corners=np.concatenate([rgb[:q,:q].reshape(-1,3),rgb[:q,w-q:].reshape(-1,3),
                                rgb[h-q:,:q].reshape(-1,3),rgb[h-q:,w-q:].reshape(-1,3)])
        bg=np.median(corners,axis=0)
        mask=(np.linalg.norm(rgb-bg,axis=2)>=BG_TOLERANCE).astype(np.uint8)*255
        cov=np.count_nonzero(mask)/max(1,mask.size)
        if cov>0.97 or cov<0.002: mask[:]=255
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((5,5),np.uint8))
    return clean_components(mask)

def outline(img: Image.Image) -> Image.Image:
    a=np.asarray(img.convert("RGBA")); gray=cv2.cvtColor(a[:,:,:3],cv2.COLOR_RGB2GRAY)
    den=cv2.bilateralFilter(gray,7,45,45)
    can=cv2.Canny(den,CANNY_LOW,CANNY_HIGH)
    block=max(3,int(ADAPTIVE_BLOCK)); block += block%2==0
    ada=cv2.adaptiveThreshold(den,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              cv2.THRESH_BINARY_INV,block,ADAPTIVE_C)
    edge=can if MODE=="canny" else ada if MODE=="adaptive" else cv2.bitwise_or(can,ada)
    edge=cv2.bitwise_and(edge,foreground(img))
    edge=cv2.morphologyEx(edge,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    edge=clean_components(edge)
    if not np.count_nonzero(edge): raise ValueError("outline kosong")
    rgba=np.zeros((*edge.shape,4),np.uint8)
    rgba[:,:,:3]=LINE_RGBA[:3]; rgba[:,:,3]=(edge>0).astype(np.uint8)*LINE_RGBA[3]
    return Image.fromarray(rgba,"RGBA")

def canonical(img: Image.Image):
    box=img.getchannel("A").getbbox()
    if box is None: raise ValueError("asset transparan")
    crop=img.crop(box); avail=round(MASTER_SIZE*0.88)
    crop.thumbnail((avail,avail),Image.Resampling.LANCZOS)
    master=Image.new("RGBA",(MASTER_SIZE,MASTER_SIZE),(0,0,0,0))
    master.alpha_composite(crop,((MASTER_SIZE-crop.width)//2,(MASTER_SIZE-crop.height)//2))
    thumb=ImageOps.contain(master,(THUMB_SIZE,THUMB_SIZE),Image.Resampling.LANCZOS)
    canvas=Image.new("RGBA",(THUMB_SIZE,THUMB_SIZE),(244,233,216,255))
    canvas.alpha_composite(thumb,((THUMB_SIZE-thumb.width)//2,(THUMB_SIZE-thumb.height)//2))
    return master,canvas.convert("RGB")

def encode_asset(name,category,png,width,height,metadata) -> bytes:
    data={"format":ASSET_FORMAT,"schema_version":ASSET_SCHEMA,"name":name,
          "category":category,"width":width,"height":height,"metadata":metadata,
          "png_base64":base64.b64encode(png).decode("ascii")}
    return json.dumps(data,ensure_ascii=False,indent=2,sort_keys=True).encode()

def validate_asset(content: bytes):
    d=json.loads(content.decode()); required={"format","schema_version","name","category",
        "width","height","metadata","png_base64"}
    assert set(d)==required and d["format"]==ASSET_FORMAT and d["schema_version"]==ASSET_SCHEMA
    raw=base64.b64decode(d["png_base64"],validate=True)
    with Image.open(io.BytesIO(raw)) as im:
        im.load(); assert im.size==(d["width"],d["height"])

def preview(original: Image.Image, result: Image.Image):
    canvas=Image.new("RGB",(840,320),(237,232,223))
    for i,img in enumerate((original,result)):
        p=Image.new("RGBA",(420,320),(255,255,255,255)); c=img.convert("RGBA")
        c.thumbnail((388,288),Image.Resampling.LANCZOS)
        p.alpha_composite(c,((420-c.width)//2,(320-c.height)//2))
        canvas.paste(p.convert("RGB"),(i*420,0))
    return canvas

In [ ]:
# ruff: noqa: E701, E702, UP017
files=discover(INPUT_DIR)
print("Gambar:",len(files))
rows=[]; items=[]; used=set()

for n,path in enumerate(files,1):
    try:
        raw=path.read_bytes()
        with Image.open(path) as src:
            src.load(); original=ImageOps.exif_transpose(src).convert("RGBA")
        result=outline(original); master,thumb=canonical(result)
        category,tags=infer(path); asset_id=safe_id(path.stem); k=2
        while asset_id in used: asset_id=safe_id(f"{path.stem}-{k}"); k+=1
        used.add(asset_id)
        name=name_of(path); master_png=png_bytes(master); thumb_png=png_bytes(thumb)
        meta={"source_path":path.as_posix().replace("/kaggle/input/",""),
              "source_sha256":hashlib.sha256(raw).hexdigest(),
              "builder":"batikcraft-kaggle-outline-generator","builder_schema":"1.0",
              "generated_at":datetime.now(timezone.utc).isoformat(),"outline_mode":MODE,
              "canny_low":CANNY_LOW,"canny_high":CANNY_HIGH,
              "background_tolerance":BG_TOLERANCE,"transparent_background":True}
        asset=encode_asset(name,category,master_png,MASTER_SIZE,MASTER_SIZE,meta)
        validate_asset(asset)
        (OUT/"png"/f"{asset_id}.png").write_bytes(master_png)
        (OUT/"thumbs"/f"{asset_id}.png").write_bytes(thumb_png)
        (OUT/"batikasset"/f"{asset_id}.batikasset").write_bytes(asset)
        preview(original,master).save(OUT/"previews"/f"{asset_id}.jpg","JPEG",quality=90)
        items.append({"id":asset_id,"name":name,"category":category,
            "file":f"assets/{asset_id}.batikasset","thumbnail":f"thumbnails/{asset_id}.png",
            "tags":list(tags),"width":MASTER_SIZE,"height":MASTER_SIZE,"metadata":meta,
            "_asset":asset,"_thumb":thumb_png})
        rows.append([1,asset_id,name,category,"|".join(tags),path.as_posix(),"ok",""])
    except Exception as exc:
        rows.append([0,"",path.stem,"","",path.as_posix(),"error",str(exc)])
    if n%25==0 or n==len(files): print(f"[{n}/{len(files)}]")

with (OUT/"summary.csv").open("w",newline="",encoding="utf-8") as f:
    import csv
    w=csv.writer(f); w.writerow(["keep","asset_id","name","category","tags","source","status","error"])
    w.writerows(rows)

if not items: raise RuntimeError("Tidak ada asset valid")
manifest={"format":PACK_FORMAT,"schema_version":PACK_SCHEMA,
 "pack":{"id":safe_id(PACK_ID),"name":PACK_NAME,"version":PACK_VERSION,
         "author":PACK_AUTHOR,"description":PACK_DESCRIPTION},
 "assets":[{k:v for k,v in x.items() if not k.startswith("_")} for x in items]}
pack=OUT/"batikpack"/f"{safe_id(PACK_ID)}.batikpack"
with zipfile.ZipFile(pack,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for x in items:
        z.writestr(x["file"],x["_asset"]); z.writestr(x["thumbnail"],x["_thumb"])
    z.writestr("manifest.json",json.dumps(manifest,ensure_ascii=False,indent=2,sort_keys=True))

with zipfile.ZipFile(pack) as z:
    loaded=json.loads(z.read("manifest.json"))
    assert loaded["format"]==PACK_FORMAT and loaded["schema_version"]==PACK_SCHEMA
    for x in loaded["assets"]: validate_asset(z.read(x["file"]))
print("VALID:",pack,"assets:",len(items))

In [ ]:
# ruff: noqa: E701, E702, UP017
# Preview hingga 8 hasil
for p in sorted((OUT/"previews").glob("*.jpg"))[:8]:
    print(p.name); display(Image.open(p))

# Zip seluruh output untuk diunduh
bundle=Path("/kaggle/working/batikcraft_outline_assets_download.zip")
if bundle.exists(): bundle.unlink()
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(OUT).as_posix())
print("Batikpack:",pack)
print("Bundle:",bundle)

## Penggunaan

1. Ubah `INPUT_DIR` ke dataset Kaggle.
2. Uji dengan `MAX_IMAGES = 20`.
3. Periksa `previews/`.
4. Setelah parameter sesuai, ubah `MAX_IMAGES = None`.
5. Instal file `.batikpack` dari folder `batikpack/` ke pustaka BatikCraftStudio.